In [ ]:
# ABPDU Automation Pipeline
This notebook contains the data harvesting and integration engine for the ABPDU Ecosystem visualization project.

### Notebook Structure:
* **Cell 1:** Live BeautifulSoup Web Scraper (with 403-bypass headers)
* **Cell 2:** YouTube Transcript API Fetcher (Baseline placeholder for Edward)
* **Cell 3:** Master Database Integration & De-duplication Engine (`export_to_pipeline`)
* **Cell 4:** Backend Pipeline Test Unit

In [11]:
# send request with a Chrome-like User-Agent to avoid 403
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Explicitly define the correct target URL
url = "https://abpdu.lbl.gov/"

# 2. Use the browser headers to bypass the 403 block
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/115.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

records = []
current_category = "Uncategorized"

# 3. Parse the page content cleanly
for element in soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "p"]):
    if element.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
        heading_text = element.get_text(strip=True)
        if heading_text:
            current_category = heading_text
    elif element.name == "p":
        paragraph_text = element.get_text(separator=" ", strip=True)
        if paragraph_text:
            records.append({
                "Data Source": url,
                "Category": current_category,
                "Content": paragraph_text
            })

# 4. Save directly to your master CSV file
df = pd.DataFrame(records, columns=["Data Source", "Category", "Content"])
df.to_csv("organized_abpdu_data.csv", index=False)
df.head()

,Data Source,Category,Content
0,https://abpdu.lbl.gov/,Designed. Developed. Scaled.,The Advanced BioProcess Development Unit (ABPD...
1,https://abpdu.lbl.gov/,Designed. Developed. Scaled.,"We enable early stage advanced biofuels, bioma..."
2,https://abpdu.lbl.gov/,Designed. Developed. Scaled.,"With experienced know-how, rigorous process op..."
3,https://abpdu.lbl.gov/,diversity of bioproducts and bioprocesses,Whatever your novel or drop-in advanced biopro...
4,https://abpdu.lbl.gov/,process development and optimization,Mitigating your risk during scale up and accel...


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from datetime import timedelta
import pandas as pd

videos_path = "youtube_videos.csv"
transcripts_path = "transcripts"

df = pd.read_csv(videos_path)
for index, id in df.iterrows():
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(id['Video ID'])

        data = {
            'Transcript': [],
            'Timestamp': [],
        }

        for line in transcript:
            text = line.text.replace("\n", " ").replace("\r", " ").replace("[Music]", "").replace("[Laughter]", "").replace("[Applause]", "")
            if not (text.startswith('"') and text.endswith('"')):
                text = f'"{text}'
            if text:
                data['Transcript'].append(text)
                data['Timestamp'].append(str(timedelta(seconds = int(line.start))))
        df = pd.DataFrame(data)
        df.to_csv(f"{transcripts_path}/{id['Video ID']}", index=False)
        print("✅ Success!")
    except Exception as e:
        print(f"❌ Transcript unavailable: {e}")

In [12]:
def export_to_pipeline(new_data, csv_path="organized_abpdu_data.csv"):
    expected_cols = ["Data Source", "Category", "Content"]

    if isinstance(new_data, pd.DataFrame):
        df_new = new_data.copy()
    elif isinstance(new_data, dict):
        df_new = pd.DataFrame([new_data])
    elif isinstance(new_data, list):
        df_new = pd.DataFrame(new_data)
    else:
        raise TypeError("new_data must be a dict, list of dicts, or a Pandas DataFrame")

    df_new = df_new.reindex(columns=expected_cols).fillna("")

    file_exists = os.path.exists(csv_path)
    if file_exists:
        df_new.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_new.to_csv(csv_path, mode="w", header=True, index=False)

    df_master = pd.read_csv(csv_path)
    before = len(df_master)
    df_cleaned = df_master.drop_duplicates()
    after = len(df_cleaned)
    df_cleaned.to_csv(csv_path, index=False)

    print(f"Master file saved to {csv_path}. Rows: {after}. Duplicates removed: {before - after}.")

In [13]:
import os
import pandas as pd

test_csv_path = "test_export_pipeline_test.csv"
if os.path.exists(test_csv_path):
    os.remove(test_csv_path)

duplicate_demo = [
    {
        "Data Source": "https://abpdu.lbl.gov/",
        "Category": "ABPDU Test",
        "Content": "Dummy ABPDU information"
    },
    {
        "Data Source": "https://abpdu.lbl.gov/",
        "Category": "ABPDU Test",
        "Content": "Dummy ABPDU information"
    }
]

# Run the pipeline with the test data
export_to_pipeline(duplicate_demo, csv_path=test_csv_path)

# Display the resulting cleaned CSV file
pd.read_csv(test_csv_path)

Master file saved to test_export_pipeline_test.csv. Rows: 1. Duplicates removed: 1.


,Data Source,Category,Content
0,https://abpdu.lbl.gov/,ABPDU Test,Dummy ABPDU information
